**Catch Images**

In [ ]:
!rm -rf test_samples/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. ضغط المجلد (اسمه سيكون my_images.zip)
!zip -r my_images.zip own

# 2. إنشاء مجلد على الدرايف باسم CV_Projects (إذا لم يكن موجوداً)
!mkdir -p /content/drive/MyDrive/CV_Projects

# 3. نقل الملف المضغوط إلى الدرايف
!cp my_images.zip /content/drive/MyDrive/CV_Projects/

print("✅ تم حفظ المجلد في الدرايف داخل: CV_Projects")

  adding: own/ (stored 0%)
  adding: own/p_dog_2.jpg (deflated 1%)
  adding: own/person_car.png (deflated 0%)
  adding: own/travel_3.jpg (deflated 1%)
  adding: own/sand_water.jpg (deflated 1%)
  adding: own/Gemini_Generated_Image_sh0oyosh0oyosh0o.png (deflated 5%)
  adding: own/11.png (deflated 5%)
  adding: own/.ipynb_checkpoints/ (stored 0%)
  adding: own/travel_1.jpg (deflated 1%)
  adding: own/travel_2.jpg (deflated 2%)
  adding: own/sand_desert.jpg (deflated 2%)
  adding: own/p_dog_1.jpg (deflated 2%)
  adding: own/car.jpg (deflated 1%)
✅ تم حفظ المجلد في الدرايف داخل: CV_Projects


In [ ]:
# 1. ربط الدرايف
from google.colab import drive
drive.mount('/content/drive')
# 2. فك ضغط الصور من الدرايف إلى الكولاب فوراً
!unzip -q /content/drive/MyDrive/CV_Projects/my_images.zip -d /content/

print("✅ الصور صارت جاهزة عندك بالـ Colab بمجلد test_samples")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ الصور صارت جاهزة عندك بالـ Colab بمجلد test_samples


**-----------------------------------------------------------------------------
**

In [ ]:
pip install ultralytics

In [ ]:
from ultralytics import YOLO

# 1. تحميل الموديل (سيقوم بتحميله تلقائياً في المرة الأولى)
# استخدمنا n للسرعة وخفة الحجم
model = YOLO('yolo11n.pt')


In [ ]:
from ultralytics import YOLO

# 1. تحميل الموديل (سيقوم بتحميله تلقائياً في المرة الأولى)
# استخدمنا n للسرعة وخفة الحجم
model = YOLO('yolo11n.pt')

# 2. تشغيل الموديل على صورة
results = model('own/person_car.png')

# 3. استخراج البيانات (المصفوفة اللي بدك اياها)
detected_objects = []

for r in results:
    for box in r.boxes:
        # الحصول على اسم العنصر (مثل: dog, person, car)
        label = model.names[int(box.cls)]
        # الحصول على نسبة الثقة (مثلاً 0.95)
        conf = float(box.conf)

        if conf > 0.0: # نأخذ فقط الأشياء المتأكد منها أكثر من 40%
            detected_objects.append(label)

# 4. تنظيف النتائج (حذف التكرار أو عدّ العناصر)
unique_objects = list(set(detected_objects))
print(f"Objects found: {unique_objects}")

# مثال للنتيجة المتوقعة: ["person", "dog", "frisbee"]


image 1/1 /content/own/person_car.png: 448x640 1 person, 166.2ms
Speed: 3.3ms preprocess, 166.2ms inference, 1.4ms postprocess per image at shape (1, 3, 448, 640)
Objects found: ['person']


In [ ]:
import json

output = {
    "image_path": "DCIM/test_image.jpg",
    "objects": list(set(detected_objects)), # قائمة فريدة
    "people_count": detected_objects.count('person'), # عد الأشخاص
    "scene": "unknown" # هاد بيحتاج موديل تاني أو تحليل لنتائج YOLO
}

print(json.dumps(output, indent=2))

{
  "image_path": "DCIM/test_image.jpg",
  "objects": [
    "person"
  ],
  "people_count": 1,
  "scene": "unknown"
}


**EMBADDINGS**

In [ ]:
# ملاحظة: ستحتاج لتنصيب sentence_transformers
# pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from PIL import Image

# 1. تحميل موديل CLIP خفيف (متوافق مع الجوال لاحقاً)
model_em = SentenceTransformer('clip-ViT-B-32')


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: sentence-transformers/clip-ViT-B-32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:

# 2. استخراج Embedding لصورة
img_emb = model_em.encode(Image.open('own/person_car.png'))

# 3. استخراج Embedding لنص (البحث)
text_emb = model_em.encode(['a photo of a dog in the park'])

print(f"Embedding shape: {img_emb.shape}")
# النتيجة غالباً مصفوفة فيها 512 رقم (هذا هو بصمة الصورة)

Embedding shape: (512,)


In [ ]:
output["embedding"] = img_emb.tolist()

In [ ]:
from sentence_transformers import SentenceTransformer, util
from PIL import Image

# 1. تحميل الموديل
model_em = SentenceTransformer('clip-ViT-B-32')

# 2. تحويل الصور لـ Embeddings (تخيل هدول صور بالاستديو عندك)
img_emb1 = model_em.encode(Image.open('own/sand_desert.jpg'))
img_emb2 = model_em.encode(Image.open('own/person_car.png'))

# 3. النص اللي كتبه المستخدم في خانة البحث
search_query = "a person behind car"
text_emb = model_em.encode(search_query)

# 4. حساب التشابه (النتيجة من 0 لـ 1)
# 1 يعني تطابق تام، 0 يعني ما في علاقة أبداً
score1 = util.cos_sim(img_emb1, text_emb)
score2 = util.cos_sim(img_emb2, text_emb)

print(f"score of 1 : {score1.item():.4f}")
print(f"score of 2 : {score2.item():.4f}")

# الصورة اللي بتجيب Score أعلى هي اللي بتطلع للمستخدم أولاً

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: sentence-transformers/clip-ViT-B-32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


score of 1 : 0.1980
score of 2 : 0.2448


**FAIS for indexing**
VECTOR SEARCH

In [ ]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. تجهيز الموديل
model_em= SentenceTransformer('clip-ViT-B-32')

# 2. تخيل هدول "بصمات" لـ 5 صور عندك بالاستديو (Embeddings)
# بالحقيقة رح تجيبهم من الموديل وتخزنهم
image_paths = ["beach.jpg", "desert.jpg", "city.jpg", "forest.jpg", "snow.jpg"]
# لنفترض عملنا encode لكل الصور وطلعوا بمصفوفة:
images_embeddings = model_em.encode([
    "a photo of a sunny beach",
    "a dry desert with dunes",
    "busy city streets",
    "green forest trees",
    "white snow mountains"
])

# 3. بناء الـ Vector Database (الفهرس)
dimension = images_embeddings.shape[1] # غالباً 512
index = faiss.IndexFlatL2(dimension)   # استخدام قياس المسافة "الأوكليدية"
index.add(np.array(images_embeddings).astype('float32')) # ضفنا الصور للفهرس

# 4. عملية البحث
query = "something very hot and dry" # المستخدم عم يبحث
query_embedding = model_em.encode([query]).astype('float32')

# ابحث عن أفضل نتيجتين (k=2)
distances, indices = index.search(query_embedding, k=2)

print("أفضل النتائج اللي لقيتها:")
for i in range(len(indices[0])):
    idx = indices[0][i]
    print(f"الصورة: {image_paths[idx]} | البعد الرياضي: {distances[0][i]:.4f}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: sentence-transformers/clip-ViT-B-32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


أفضل النتائج اللي لقيتها:
الصورة: city.jpg | البعد الرياضي: 39.3219
الصورة: beach.jpg | البعد الرياضي: 44.0503


In [ ]:
!pip install faiss-cpu
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. تجهيز الموديل
model_em = SentenceTransformer('clip-ViT-B-32')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
from PIL import Image
import numpy as np
import os

# مصفوفة لتخزين مسارات الصور الحقيقية
image_paths = ["own/travel_1.jpg", "own/p_dog_1.jpg", "own/11.png"]

# تحويل الصور الحقيقية إلى Embeddings
# الموديل هون عم "يشوف" الصور ويحولها لأرقام
real_images_embeddings = []
for path in image_paths:
    img = Image.open(path)
    embedding = model_em.encode(img) # هون السحر: الموديل عم يحلل بكسلات الصورة
    real_images_embeddings.append(embedding)
# تحويل القائمة لمصفوفة numpy (ضروري لـ FAISS)
embeddings_array = np.array(real_images_embeddings).astype('float32')

# 4. **هون تعريف الـ index اللي كان ناقصك**
dimension = embeddings_array.shape[1] # غالباً 512 في موديل CLIP
index = faiss.IndexFlatIP(dimension) # IndexFlatIP بيستخدم للتشابه (Inner Product)
index.add(embeddings_array) # هون ضفنا بصمات الصور للفهرس

In [ ]:
# حفظ الفهرس في ملف حقيقي
faiss.write_index(index, "photos_index.faiss")

# وحفظ مسارات الصور بملف نصي أو JSON
# لأن الفهرس بيحفظ "أرقام" بس، ما بيعرف شو اسم الصورة
import json
with open("image_mapping.json", "w") as f:
    json.dump(image_paths, f)

print("تمت عملية الفهرسة بنجاح!")

تمت عملية الفهرسة بنجاح!


In [ ]:
search_query = "a p" # الجملة اللي بدك تبحث عنها
query_embedding = model_em.encode([search_query]).astype('float32')

# البحث عن أفضل نتيجتين (k=2)
# D هي المسافة (التشابه)، I هي أرقام الصور في الفهرس
D, I = index.search(query_embedding, k=2)

print("\nنتائج البحث:")
for i in range(len(I[0])):
    idx = I[0][i]
    print(f"الصورة: {image_paths[idx]} | نسبة التشابه: {D[0][i]:.4f}")


نتائج البحث:
الصورة: own/p_dog_1.jpg | نسبة التشابه: 28.5185
الصورة: own/travel_1.jpg | نسبة التشابه: 21.4657


In [ ]:
# التصدير لصيغة TFLite مع تفعيل الـ Quantization لتقليل الحجم
model.export(format='tflite', int8=True)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'yolo11n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.4 MB)
TensorFlow SavedModel: collecting INT8 calibration images from 'data=coco8.yaml'

WARNING ⚠️ Dataset 'coco8.yaml' images not found, missing path '/content/datasets/coco8/images/val'
Unzipping /content/datasets/coco8.zip to /content/datasets/coco8...: 100% ━━━━━━━━━━━━ 25/25 1.6Kfiles/s 0.0s
Dataset download success ✅ (0.3s), saved to /content/datasets

Fast image access ✅ (ping: 0.0±0.0 ms, read: 734.4±635.2 MB/s, size: 54.0 KB)
Scanning /content/datasets/coco8/labels/val... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 893.9it/s 0.0s

'yolo11n_saved_model/yolo11n_int8.tflite'

In [ ]:
from ultralytics import YOLO
from PIL import Image

# 1. تحميل موديل TFLite الذي تم تصديره
# تأكد أن المسار صحيح
tflite_model_path = '/content/yolo11n_saved_model/yolo11n_float32.tflite'
model_tflite = YOLO(tflite_model_path)

# 2. حدد مسار الصورة التي تريد اختبارها مع موديل TFLite
image_to_test_tflite = 'own/11.png' # يمكنك تغيير هذا المسار

# 3. تشغيل الموديل على الصورة المحددة
# لاحظ أن YOLO v8 تتعامل مع TFLite بنفس طريقة PyTorch model (للتنبؤ)
results_tflite = model_tflite(image_to_test_tflite)

# 4. استخراج البيانات (المصفوفة التي تحتوي على العناصر المكتشفة)
detected_objects_tflite = []

for r in results_tflite:
    for box in r.boxes:
        # الحصول على اسم العنصر
        # أسماء الفئات عادة ما تكون مدمجة داخل ملف TFLite أو يمكن الوصول إليها من الموديل الأصلي
        label = model_tflite.names[int(box.cls)]
        # الحصول على نسبة الثقة
        conf = float(box.conf)

        if conf > 0.1: # نأخذ فقط الأشياء المتأكد منها أكثر من 10%
            detected_objects_tflite.append(label)

# 5. تنظيف النتائج (حذف التكرار)
unique_objects_tflite = list(set(detected_objects_tflite))

print(f"تم العثور على الكائنات التالية في الصورة '{image_to_test_tflite}' باستخدام موديل TFLite: {unique_objects_tflite}")


WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading /content/yolo11n_saved_model/yolo11n_float32.tflite for TensorFlow Lite inference...

image 1/1 /content/own/11.png: 640x640 17 persons, 1 tie, 1 dining table, 159.2ms
Speed: 4.1ms preprocess, 159.2ms inference, 2.3ms postprocess per image at shape (1, 3, 640, 640)
تم العثور على الكائنات التالية في الصورة 'own/11.png' باستخدام موديل TFLite: ['dining table', 'person', 'tie']


.

.

**OPEN CLIP**

In [ ]:
pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00


In [ ]:
import torch
import open_clip
from PIL import Image
import torch.nn.functional as F

# 1. تحميل الموديل الخفيف (MobileCLIP S1)
# هذا الموديل حجمه حوالي 40-50 ميجابايت فقط!
model, _, preprocess = open_clip.create_model_and_transforms('MobileClip_s1', pretrained='apple')
tokenizer = open_clip.get_tokenizer('MobileClip-s1')

# 2. تجهيز الصور (نفس صورك السابقة)
image_paths = ["own/sand_water.jpg", "own/travel_1.jpg"]
images = [preprocess(Image.open(p)).unsqueeze(0) for p in image_paths]
image_stack = torch.cat(images)

# 3. تجهيز النص
search_query = "a sand desert"
text = tokenizer([search_query])

# 4. استخراج الـ Embeddings (بوضع الـ Inference)
with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model.encode_image(image_stack)
    text_features = model.encode_text(text)

    # عمل Normalization (ضروري جداً في CLIP للمقارنة)
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    # حساب التشابه (Cosine Similarity)
    # النتيجة هنا هي مصفوفة، نأخذ القيم منها
    similarities = (text_features @ image_features.T).cpu().numpy()[0]

print(f"التشابه مع الصورة الأولى: {similarities[0]:.4f}")
print(f"التشابه مع الصورة الثانية: {similarities[1]:.4f}")

RuntimeError: Model config for 'MobileClip_s1' not found in built-ins. Available: ['coca_base', 'coca_roberta-ViT-B-32', 'coca_ViT-B-32', 'coca_ViT-L-14', 'convnext_base', 'convnext_base_w', 'convnext_base_w_320', 'convnext_large', 'convnext_large_d', 'convnext_large_d_320', 'convnext_small', 'convnext_tiny', 'convnext_xlarge', 'convnext_xxlarge', 'convnext_xxlarge_320', 'EVA01-g-14', 'EVA01-g-14-plus', 'EVA02-B-16', 'EVA02-E-14', 'EVA02-E-14-plus', 'EVA02-L-14', 'EVA02-L-14-336', 'MobileCLIP2-B', 'MobileCLIP2-L-14', 'MobileCLIP2-S0', 'MobileCLIP2-S2', 'MobileCLIP2-S3', 'MobileCLIP2-S4', 'MobileCLIP-B', 'MobileCLIP-S1', 'MobileCLIP-S2', 'mt5-base-ViT-B-32', 'mt5-xl-ViT-H-14', 'nllb-clip-base', 'nllb-clip-base-siglip', 'nllb-clip-large', 'nllb-clip-large-siglip', 'PE-Core-B-16', 'PE-Core-bigG-14-448', 'PE-Core-L-14-336', 'PE-Core-S-16-384', 'PE-Core-T-16-384', 'RN50', 'RN50-quickgelu', 'RN50x4', 'RN50x4-quickgelu', 'RN50x16', 'RN50x16-quickgelu', 'RN50x64', 'RN50x64-quickgelu', 'RN101',

.




```
`# This is formatted as code`
**BlIB**```



In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

# 1. تحميل الموديل والمعالج (نسخة base خفيفة)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# 2. تحميل الصورة
img_path = "own/travel_1.jpg" # ضع مسار صورتك هنا
raw_image = Image.open(img_path).convert('RGB')

# 3. معالجة الصورة وتوليد الوصف
inputs = processor(raw_image, return_tensors="pt")
out = model.generate(**inputs)

# 4. طباعة الوصف
caption = processor.decode(out[0], skip_special_tokens=True)
print(f"وصف الصورة: {caption}")

# مثال للناتج: "a person sitting inside a car on a sunny day"

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

وصف الصورة: a group of people walking down a street


In [ ]:
# 2. تحميل الصورة
img_path = "own/person_car.png" # ضع مسار صورتك هنا
raw_image = Image.open(img_path).convert('RGB')

# 3. معالجة الصورة وتوليد الوصف
inputs = processor(raw_image, return_tensors="pt")
out = model.generate(**inputs)

# 4. طباعة الوصف
caption = processor.decode(out[0], skip_special_tokens=True)
print(f"وصف الصورة: {caption}")

# مثال للناتج: "a person sitting inside a car on a sunny day"

وصف الصورة: man driving a car
